In [0]:
df_silver = spark.table("prf_acidentes.silver.acidentes_limpo")
print("Total de linhas:", df_silver.count())

In [0]:
from pyspark.sql import functions as F

dim_tempo = (
    df_silver
    .select("data_inversa", "dia_semana", "horario", "hora", "fase_dia", "flag_fim_de_semana", "_ano_referencia")
    .distinct()
    .withColumn("id_tempo", F.monotonically_increasing_id())
    .withColumn("ano", F.year("data_inversa"))
    .withColumn("mes", F.month("data_inversa"))
    .withColumn("dia", F.dayofmonth("data_inversa"))
    .select(
        "id_tempo", "data_inversa", "ano", "mes", "dia",
        "dia_semana", "horario", "hora", "fase_dia", "flag_fim_de_semana"
    )
)

dim_tempo.write.format("delta").mode("overwrite").saveAsTable("prf_acidentes.gold.dim_tempo")

print("dim_tempo criada. Total de linhas:", dim_tempo.count())

In [0]:
from pyspark.sql import functions as F

dim_local = (
    df_silver
    .select("uf", "br", "km", "municipio", "regional", "delegacia", "uop", "latitude", "longitude")
    .distinct()
    .withColumn("id_local", F.monotonically_increasing_id())
    .select(
        "id_local", "uf", "br", "km", "municipio",
        "regional", "delegacia", "uop", "latitude", "longitude"
    )
)

dim_local.write.format("delta").mode("overwrite").saveAsTable("prf_acidentes.gold.dim_local")

print("dim_local criada. Total de linhas:", dim_local.count())

In [0]:
from pyspark.sql import functions as F

dim_causa = (
    df_silver
    .select("causa_acidente", "tipo_acidente")
    .distinct()
    .withColumn("id_causa", F.monotonically_increasing_id())
    .select("id_causa", "causa_acidente", "tipo_acidente")
)

dim_causa.write.format("delta").mode("overwrite").saveAsTable("prf_acidentes.gold.dim_causa")

print("dim_causa criada. Total de linhas:", dim_causa.count())

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Criação da dimensão
dim_condicao = (
    df_silver
    .select(
        "condicao_metereologica",
        "tipo_pista",
        "tracado_via",
        "uso_solo",
        "sentido_via"
    )
    .distinct()
    .withColumn(
        "id_condicao",
        F.row_number().over(
            Window.orderBy(
                "condicao_metereologica",
                "tipo_pista",
                "tracado_via",
                "uso_solo",
                "sentido_via"
            )
        )
    )
    .select(
        "id_condicao",
        "condicao_metereologica",
        "tipo_pista",
        "tracado_via",
        "uso_solo",
        "sentido_via"
    )
)

# Remove tabela anterior, caso exista
spark.sql("DROP TABLE IF EXISTS prf_acidentes.gold.dim_condicao")

# Persistência em Delta
dim_condicao.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("prf_acidentes.gold.dim_condicao")

print(
    "dim_condicao criada. Total de linhas:",
    dim_condicao.count()
)


In [0]:
dim_classificacao = (
    df_silver
    .select("classificacao_acidente")
    .distinct()
    .withColumn("id_classificacao", F.monotonically_increasing_id())
    .select("id_classificacao", "classificacao_acidente")
)

dim_classificacao.write.format("delta").mode("overwrite").saveAsTable("prf_acidentes.gold.dim_classificacao")

print("dim_classificacao criada. Total de linhas:", dim_classificacao.count())

In [0]:
from pyspark.sql import functions as F

for coluna in ["condicao_metereologica", "tipo_pista", "tracado_via", "uso_solo", "sentido_via"]:
    print(f"--- {coluna} ---")
    df_silver.select(coluna).distinct().orderBy(coluna).show(50, truncate=False)

In [0]:
from pyspark.sql import functions as F

valores_unicos = (
    df_silver
    .select(F.explode(F.split(F.col("tracado_via"), ";")).alias("valor"))
    .select(F.trim(F.col("valor")).alias("valor"))
    .distinct()
    .orderBy("valor")
)

valores_unicos.show(50, truncate=False)

In [0]:
from pyspark.sql import functions as F

mapa_flags = {
    "flag_aclive": "Aclive",
    "flag_curva": "Curva",
    "flag_declive": "Declive",
    "flag_desvio_temporario": "Desvio Temporário",
    "flag_em_obras": "Em Obras",
    "flag_intersecao_vias": "Interseção de Vias",
    "flag_ponte": "Ponte",
    "flag_reta": "Reta",
    "flag_retorno_regulamentado": "Retorno Regulamentado",
    "flag_rotatoria": "Rotatória",
    "flag_tunel": "Túnel",
    "flag_viaduto": "Viaduto",
}

df_silver_flags = df_silver
for nome_flag, valor_busca in mapa_flags.items():
    df_silver_flags = df_silver_flags.withColumn(
        nome_flag,
        F.array_contains(F.split(F.col("tracado_via"), ";"), valor_busca)
    )

df_silver_flags.select("tracado_via", *mapa_flags.keys()).show(10, truncate=False)

In [0]:
dim_condicao = (
    df_silver_flags
    .select(
        "condicao_metereologica", "tipo_pista", "uso_solo", "sentido_via",
        "flag_aclive", "flag_curva", "flag_declive", "flag_desvio_temporario",
        "flag_em_obras", "flag_intersecao_vias", "flag_ponte", "flag_reta",
        "flag_retorno_regulamentado", "flag_rotatoria", "flag_tunel", "flag_viaduto"
    )
    .distinct()
    .withColumn("id_condicao", F.monotonically_increasing_id())
)

dim_condicao.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("prf_acidentes.gold.dim_condicao")

print("dim_condicao recriada. Total de linhas:", dim_condicao.count())

In [0]:
dim_tempo = spark.table("prf_acidentes.gold.dim_tempo")
dim_local = spark.table("prf_acidentes.gold.dim_local")
dim_causa = spark.table("prf_acidentes.gold.dim_causa")
dim_condicao = spark.table("prf_acidentes.gold.dim_condicao")
dim_classificacao = spark.table("prf_acidentes.gold.dim_classificacao")

print("Dimensões carregadas.")

In [0]:
colunas_tempo = ["data_inversa", "dia_semana", "horario", "hora", "fase_dia", "flag_fim_de_semana"]
colunas_local = ["uf", "br", "km", "municipio", "regional", "delegacia", "uop", "latitude", "longitude"]
colunas_causa = ["causa_acidente", "tipo_acidente"]
colunas_condicao = [
    "condicao_metereologica", "tipo_pista", "uso_solo", "sentido_via",
    "flag_aclive", "flag_curva", "flag_declive", "flag_desvio_temporario",
    "flag_em_obras", "flag_intersecao_vias", "flag_ponte", "flag_reta",
    "flag_retorno_regulamentado", "flag_rotatoria", "flag_tunel", "flag_viaduto"
]
colunas_classificacao = ["classificacao_acidente"]

fato_temp = df_silver_flags.join(dim_tempo, on=colunas_tempo, how="left")
fato_temp = fato_temp.join(dim_local, on=colunas_local, how="left")
fato_temp = fato_temp.join(dim_causa, on=colunas_causa, how="left")
fato_temp = fato_temp.join(dim_condicao, on=colunas_condicao, how="left")
fato_temp = fato_temp.join(dim_classificacao, on=colunas_classificacao, how="left")

print("Joins concluídos.")

In [0]:
fato_acidente = (
    fato_temp
    .select(
        "id",
        "id_tempo", "id_local", "id_causa", "id_condicao", "id_classificacao",
        "pessoas", "mortos", "feridos_leves", "feridos_graves", "ilesos", "ignorados", "feridos", "veiculos",
        "_ano_referencia"
    )
    .withColumnRenamed("id", "id_acidente")
)

print("Total de linhas na fato_acidente:", fato_acidente.count())

In [0]:
fato_acidente.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("prf_acidentes.gold.fato_acidente")

print("Tabela fato_acidente criada com sucesso!")

In [0]:
display(spark.sql("SELECT * FROM prf_acidentes.gold.fato_acidente LIMIT 10"))